In [1]:
import os

In [2]:
%pwd

'd:\\Data Science\\project Series\\End_to_end_ReD_Wine_Quality_Mlops_Project\\research'

In [3]:
os.chdir('../')

In [4]:
%pwd

'd:\\Data Science\\project Series\\End_to_end_ReD_Wine_Quality_Mlops_Project'

In [5]:
from pathlib import Path
from dataclasses import dataclass

@dataclass(frozen=True)
class DataTransformationconfig:
    root_dir:Path
    data_path:Path

In [6]:
from WineQuality_Project.utils.common import read_yaml,create_directories
from WineQuality_Project.constants import *

In [7]:
class ConfigManager:
    def __init__(self,
                 config_filepath: Path = CONFIG_FILE_PATH,
                 params_filepath: Path = PARAMS_FILE_PATH,
                 schema_filepath: Path = SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        create_directories([Path(self.config['artifact_root'])])

    def get_data_transformation_config(self) -> DataTransformationconfig:
        config = self.config["data_transformation"]   # ← FIXED (dict access)

        root_dir = Path(config["root_dir"])
        data_path = Path(config["data_path"])

        create_directories([root_dir])  # ← FIXED

        data_transformation_config = DataTransformationconfig(
            root_dir=root_dir,
            data_path=data_path
        )

        return data_transformation_config


In [8]:
import os
import pandas as pd
import numpy as np
import logging
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import joblib

In [9]:
class DataTransformation:
    def __init__(self,config:DataTransformationconfig):
        self.config=config
    
    def initiate_data_transformation(self):
        logging.info("🚀 Starting Data Transformation Stage...")

        try:
            df = pd.read_csv(self.config.data_path)
            logging.info(f"Loaded dataset: {self.config.data_path}")

            # --------------------------------------
            # 1️⃣ Separate features and target
            # --------------------------------------
            X = df.drop(columns=["quality"])
            y = df["quality"]

            # --------------------------------------
            # 2️⃣ Train–Test Split
            # --------------------------------------
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.2, random_state=42
            )

            logging.info("Train-test split completed")

            scalar=StandardScaler()
            X_train_scaled=scalar.fit_transform(X_train)
            X_test_scaled=scalar.transform(X_test)

            logging.info("Feature scaling completed")
            train_path = Path(self.config.root_dir, "train.npy")
            test_path = Path(self.config.root_dir, "test.npy")
            ytrain_path = Path(self.config.root_dir, "y_train.npy")
            ytest_path = Path(self.config.root_dir, "y_test.npy")
            scaler_path = Path(self.config.root_dir, "scaler.pkl")

            np.save(train_path, X_train_scaled)
            np.save(test_path, X_test_scaled)
            np.save(ytrain_path, y_train)
            np.save(ytest_path, y_test)
            

            # save scaler
            import joblib
            joblib.dump(scalar, scaler_path)

            logging.info(f"Artifacts saved inside: {self.config.root_dir}")

            return {
                "X_train": train_path,
                "X_test": test_path,
                "y_train": ytrain_path,
                "y_test": ytest_path,
                "scaler": scaler_path
            }
        except Exception as e:
             logging.error(f"Data Transformation failed: {e}")
             raise e

In [10]:
try:
    config = ConfigManager()
    
    data_transformation_config = config.get_data_transformation_config()
    
    data_transformation = DataTransformation(config=data_transformation_config)
    
    artifacts = data_transformation.initiate_data_transformation()

    print("✔ Data Transformation Completed Successfully!")
    print(artifacts)

except Exception as e:
    print(" Error in Data Transformation Stage")
    raise e


[2025-12-12 04:47:31] [INFO] WineQualityLogger - YAML file: config\config.yml loaded successfully
[2025-12-12 04:47:31] [INFO] WineQualityLogger - YAML file: params.yaml loaded successfully
[2025-12-12 04:47:31] [INFO] WineQualityLogger - YAML file: schema.yaml loaded successfully
[2025-12-12 04:47:31] [INFO] WineQualityLogger - Directory created at: artifacts
[2025-12-12 04:47:31] [INFO] WineQualityLogger - Directory created at: artifacts\data_transformation


✔ Data Transformation Completed Successfully!
{'X_train': WindowsPath('artifacts/data_transformation/train.npy'), 'X_test': WindowsPath('artifacts/data_transformation/test.npy'), 'y_train': WindowsPath('artifacts/data_transformation/y_train.npy'), 'y_test': WindowsPath('artifacts/data_transformation/y_test.npy'), 'scaler': WindowsPath('artifacts/data_transformation/scaler.pkl')}
